# 01 — Data Understanding + Causal DAG

Goals:
- Load synthetic field panel and understand structure
- Treatment assignment mechanism — visualize how newer units got variant B (selection bias)
- Confounder distributions: region, install crew, install date vs. design variant
- Draw the causal DAG explicitly
- Document confounders that naive comparison misses

Key output: causal DAG image for the Quarto report

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys; sys.path.insert(0, '..')
from src.data_generator import main as gen_main, TRUE_HAZARD_RATIO
from pathlib import Path

%matplotlib inline
sns.set_theme(style='darkgrid')

In [ ]:
# Generate panel if not on disk
if not Path('../data/field_panel.csv').exists():
    import os; os.chdir('..'); gen_main(); os.chdir('notebooks')

df = pd.read_csv('../data/field_panel.csv', parse_dates=['install_date'])
print(f'Units: {len(df):,} | Failures: {df.failure_event.sum():,} ({df.failure_event.mean()*100:.1f}%)')
df.head(3)

In [ ]:
# Treatment assignment over time — should show newer units skewing toward variant B
df['install_month'] = df.install_date.dt.to_period('M')
trt_rate = df.groupby('install_month').apply(lambda x: (x.design_variant=='B').mean())
trt_rate.plot(figsize=(10,3), title='Fraction receiving Variant B by install month')
plt.ylabel('Fraction B'); plt.xlabel('Install month')
plt.tight_layout(); plt.show()
print('This is the selection bias — newer units more likely to get variant B')

In [ ]:
# Failure rate by region and variant (shows confounding)
pivot = df.groupby(['region', 'design_variant'])['failure_event'].mean().unstack() * 100
print(pivot.round(1))
print('\nNote: regional difference in failure rate is a confounder')

In [ ]:
# Causal DAG using DoWhy
import dowhy
from dowhy import CausalModel

model = CausalModel(
    data=df,
    treatment='design_variant',
    outcome='failure_event',
    common_causes=['region', 'install_crew', 'install_date'],
)
model.view_model()
# Save: figures/causal_dag.png
print(f'True causal effect to recover: HR = {TRUE_HAZARD_RATIO} (variant B reduces hazard by {(1-TRUE_HAZARD_RATIO)*100:.0f}%)')

In [ ]:
# Naive failure rate comparison (WRONG)
nv = df.groupby('design_variant').failure_event.mean() * 100
print('=== Naive comparison (WRONG) ===')
print(nv.round(2))
bias = (nv['B'] - nv['A']) / nv['A'] * 100
print(f'Naive estimate: {bias:+.1f}% — this is confounded')
print(f'True effect: -15.0%')
print(f'Naive bias: {bias - (-15.0):+.1f} percentage points')